# State estimation: measurements, GENCO and the WLS baseline

This notebook simulates SCADA measurements for one IEEE 14-bus scenario, then
estimates the state with a GENCO model and with the WLS baseline.

It uses the scenarios in `tests/data`. Run it from the repository root.
Set `MODEL_PATH` to a `best_model_state_dict.pt` and `STATS_PATH` to its
`normalizer_stats.pt` (both in the artifacts of a training run). With
`MODEL_PATH = None` the notebook uses an untrained model, so the GENCO plot
only shows the interface.

In [ ]:
from copy import deepcopy

import torch
import yaml
from matplotlib import pyplot as plt
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader

from gridfm_graphkit.datasets.globals import (
    BS,
    GS,
    VA_H,
    VA_OUT,
    VM_H,
    VM_OUT,
    YFF_TT_R,
    YFT_TF_I,
)
from gridfm_graphkit.datasets.normalizers import HeteroDataMVANormalizer
from gridfm_graphkit.datasets.task_transforms import StateEstimationTransforms
from gridfm_graphkit.io.param_handler import NestedNamespace, get_task
from gridfm_graphkit.models.pandapower_se import PandapowerSE

CONFIG = "scripts/state_estimation/configs/case14_ieee_base_0.yaml"
DATA = "tests/data/case14_ieee"
MODEL_PATH = None
STATS_PATH = None

with open(CONFIG) as f:
    args = NestedNamespace(**yaml.safe_load(f))

normalizer = HeteroDataMVANormalizer(args)
if STATS_PATH is None:
    normalizer.fit(f"{DATA}/raw", list(range(72)))
else:
    normalizer.fit_from_dict(torch.load(STATS_PATH, weights_only=True)["case14_ieee"])

data = HeteroData.from_dict(
    torch.load(f"{DATA}/processed/data_index_0.pt", weights_only=True)
)
normalizer.transform(data)

## Measurement noise and outliers

Measured voltage magnitudes get 1% relative Gaussian noise. Outliers are shifted
by 3 standard deviations. The histograms show the relative measurement error for
several outlier rates.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3), sharey=True)
for ax, outlier_ratio in zip(axes, [0.0, 0.2, 0.4, 0.8]):
    a = deepcopy(args)
    a.task.measurements.vm.outlier_ratio = outlier_ratio
    out = StateEstimationTransforms(a)(deepcopy(data))
    measured = ~out.mask_dict["bus"][:, VM_H]
    error = (out["bus"].x[measured, VM_H] - out["bus"].y[measured, VM_H]) / out[
        "bus"
    ].y[measured, VM_H]
    ax.hist(error, bins=20)
    ax.set_title(f"p_outlier = {outlier_ratio}")
    ax.set_xlabel("relative Vm error")

## GENCO estimate

In [ ]:
task = get_task(args, [normalizer])
if MODEL_PATH is not None:
    task.load_state_dict(torch.load(MODEL_PATH, map_location="cpu", weights_only=True))
task.eval()

batch = next(
    iter(DataLoader([StateEstimationTransforms(args)(deepcopy(data))], batch_size=1))
)
wls_batch = deepcopy(batch)
with torch.no_grad():
    output, _ = task.shared_step(batch)

## WLS estimate

The WLS baseline takes powers in MW and admittances in p.u. on the case base, so
the batch is denormalized first. `StateEstimationTask.run_wls` does the same.

In [ ]:
normalizer.inverse_transform(wls_batch)
normalizer.inverse_std(wls_batch)
wls_batch.x_dict["bus"][:, VA_H] *= 180 / torch.pi
wls_batch.edge_attr_dict[("bus", "connects", "bus")][:, YFF_TT_R : YFT_TF_I + 1] /= (
    normalizer.baseMVA_orig
)
wls_batch.x_dict["bus"][:, [GS, BS]] /= normalizer.baseMVA_orig

wls = PandapowerSE(args)(
    x_dict=wls_batch.x_dict,
    edge_index_dict=wls_batch.edge_index_dict,
    edge_attr_dict=wls_batch.edge_attr_dict,
    mask_dict=wls_batch.mask_dict,
    baseMVA_orig=normalizer.baseMVA_orig,
)
print(
    "WLS converged:", wls["success"], "| measurements removed:", wls["bad_data_cycles"]
)

In [ ]:
true_vm = batch.y_dict["bus"][:, VM_H]
true_va = batch.y_dict["bus"][:, VA_H] * 180 / torch.pi

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(true_vm, output["bus"][:, VM_OUT], label="GENCO")
axes[1].scatter(true_va, output["bus"][:, VA_OUT] * 180 / torch.pi, label="GENCO")
if wls["success"]:
    axes[0].scatter(true_vm, wls["bus"][:, VM_OUT], marker="x", label="WLS")
    axes[1].scatter(true_va, wls["bus"][:, VA_OUT], marker="x", label="WLS")
axes[0].set(xlabel="true Vm (p.u.)", ylabel="estimate")
axes[1].set(xlabel="true Va (degrees)", ylabel="estimate")
axes[0].legend()